## Course Assignment 4

In [17]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
sns.set()

ticker_aapl = "AAPL"
df_aapl = yf.download(ticker_aapl, start="2010-01-01", end="2019-12-31")

ticker_sp500 = "^GSPC"
df_sp500 = yf.download(ticker_sp500, start="2010-01-01", end="2019-12-31")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [18]:
df_aapl.columns = df_aapl.columns.droplevel(1)
df_sp500.columns = df_sp500.columns.droplevel(1)

In [19]:
prices = pd.DataFrame(index=df_aapl.index)
prices['AAPL'] = df_aapl['Close']
prices['SP500'] = df_sp500['Close']

In [21]:
prices.head()

,AAPL,SP500
Date,,
2010-01-04,6.412382,1132.989990
2010-01-05,6.423469,1136.520020
2010-01-06,6.321296,1137.140015
2010-01-07,6.309609,1141.689941
2010-01-08,6.351558,1144.979980


In [26]:
def getBeta(prices):
    """
    prices: dataframe with 2 columns
        prices.columns[0]: price of the stock
        prices.columns[1]: price of the market
    """
    returns = prices.pct_change(1)
    returns.dropna(inplace=True)
    deviations = returns - returns.mean()
    product_deviations = deviations.iloc[:,0] * deviations.iloc[:,1]
    covariance = product_deviations.sum() / (len(product_deviations) - 1)
    ssd_market = deviations.iloc[:,1] * deviations.iloc[:,1]
    variance_market = ssd_market.sum() / (len(ssd_market) - 1)
    print(covariance, variance_market)
    return covariance / variance_market

In [27]:
print(getBeta(prices))

8.976228609478892e-05 8.654704012491843e-05
1.0371502707109306


In [29]:
returns = prices.pct_change(1)
returns.dropna(inplace=True)
cov_matrix = np.cov(returns['AAPL'], returns['SP500'])
print(cov_matrix)

[[2.63175955e-04 8.97622861e-05]
 [8.97622861e-05 8.65470401e-05]]


In [30]:
beta_matrix = cov_matrix[0][1] / cov_matrix[1][1]
print(beta_matrix)

1.0371502707109306


In [32]:
beta_lin = linregress(y=returns['AAPL'], x=returns['SP500'])[0]
print(beta_lin)

1.0371502707109306


Beta estimates may differ because np.cov() and linregress may assume ddof = 0, while we are using ddof = 1 in getBeta(prices)

Beta will be identical only if the same ddof is constantly used across all calculations